In [1]:
!pip uninstall -y triton bitsandbytes transformers tokenizers -q

!pip install -q \
    triton==2.3.0 \
    bitsandbytes==0.42.0 \
    transformers==4.41.2 \
    tokenizers==0.19.1 \
    peft==0.11.1 \
    trl==0.8.6 \
    datasets==2.19.0 \
    accelerate==0.30.1 \
    huggingface_hub==0.23.2 \
    sentencepiece

print("✅ Done — DO NOT run any other cell yet")

✅ Done — DO NOT run any other cell yet


In [2]:
!pip install -q transformers==4.41.2 peft==0.11.1 trl==0.8.6 \
    bitsandbytes==0.43.1 datasets==2.19.0 accelerate==0.30.1 \
    huggingface_hub==0.23.2 sentencepiece

print("✅ Done")

✅ Done


In [ ]:
from google.colab import userdata
import os
from huggingface_hub import login

# Retrieve the secret from Colab's secret manager
hf_token = userdata.get('HF_TOKEN')

if hf_token:
    # Use the token to login
    login(token=hf_token)
    # Also set it as an environment variable (many libraries look for this)
    os.environ["HF_TOKEN"] = hf_token

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # ← 3B fits on T4 easily

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False

print("✅ Model loaded")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded
GPU memory: 4.66 GB


In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [7]:
!wget https://raw.githubusercontent.com/harshvardhan1448/cloudstok_inventory_agent/main/finetuning/dataset_sample.jsonl

# Verify it downloaded
!cat dataset_sample.jsonl

--2026-04-02 14:43:37--  https://raw.githubusercontent.com/harshvardhan1448/cloudstok_inventory_agent/main/finetuning/dataset_sample.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 732 [text/plain]
Saving to: ‘dataset_sample.jsonl’

dataset_sample.json 100%[===================>]     732  --.-KB/s    in 0s      

2026-04-02 14:43:37 (27.4 MB/s) - ‘dataset_sample.jsonl’ saved [732/732]

{"instruction":"How many units of SKU-1001 are in stock?","input":"","output":"SELECT quantity FROM products WHERE sku = 'SKU-1001'"}
{"instruction":"List all products below reorder point","input":"","output":"SELECT sku, name, quantity, reorder_point FROM products WHERE quantity <= reorder_point"}
{"instruction":"Show transaction history for SKU-1004","input":"","output"

In [8]:
from datasets import load_dataset, Dataset
import json

# Load your custom inventory examples
with open("dataset_sample.jsonl", "r") as f:
    custom_data = [json.loads(line) for line in f]

# Load Spider dataset for extra text-to-SQL coverage
spider = load_dataset("spider", split="train[:300]", trust_remote_code=True)
spider_formatted = [
    {"instruction": row["question"], "output": row["query"]}
    for row in spider
]

# Combine both
all_data = custom_data + spider_formatted
print(f"Custom examples: {len(custom_data)}")
print(f"Spider examples: {len(spider_formatted)}")
print(f"Total: {len(all_data)}")

# Format into instruction template
def format_prompt(example):
    text = f"""### Instruction:
Convert the following natural language query to SQL for an inventory database with tables:
- products (id, sku, name, description, quantity, reorder_point, supplier_id, unit_price)
- transactions (id, sku, change, reason, timestamp)
- suppliers (id, name, contact_email)

### Query:
{example['instruction']}

### SQL:
{example['output']}{tokenizer.eos_token}"""
    return {"text": text}

dataset = Dataset.from_list(all_data)
dataset = dataset.map(format_prompt)

print("\n✅ Dataset ready")
print("\nSample prompt:")
print(dataset[0]["text"])

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Custom examples: 5
Spider examples: 300
Total: 305


Map:   0%|          | 0/305 [00:00<?, ? examples/s]


✅ Dataset ready

Sample prompt:
### Instruction:
Convert the following natural language query to SQL for an inventory database with tables:
- products (id, sku, name, description, quantity, reorder_point, supplier_id, unit_price)
- transactions (id, sku, change, reason, timestamp)
- suppliers (id, name, contact_email)

### Query:
How many units of SKU-1001 are in stock?

### SQL:
SELECT quantity FROM products WHERE sku = 'SKU-1001'</s>


In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_steps=100,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

# Print loss curve
loss_log = [(log["step"], log["loss"]) for log in trainer.state.log_history if "loss" in log]
print("\n📉 Loss Curve:")
for step, loss in loss_log:
    print(f"  Step {step:3d} → loss = {loss:.4f}")

Map:   0%|          | 0/305 [00:00<?, ? examples/s]

🚀 Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
5,2.223700
10,1.204000
15,0.658800
20,0.541200
25,0.454600
30,0.414300
35,0.397600
40,0.353800
45,0.308800
50,0.306400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


✅ Training complete!

📉 Loss Curve:
  Step   5 → loss = 2.2237
  Step  10 → loss = 1.2040
  Step  15 → loss = 0.6588
  Step  20 → loss = 0.5412
  Step  25 → loss = 0.4546
  Step  30 → loss = 0.4143
  Step  35 → loss = 0.3976
  Step  40 → loss = 0.3538
  Step  45 → loss = 0.3088
  Step  50 → loss = 0.3064
  Step  55 → loss = 0.2780
  Step  60 → loss = 0.2508
  Step  65 → loss = 0.2400
  Step  70 → loss = 0.2354
  Step  75 → loss = 0.2263
  Step  80 → loss = 0.1853
  Step  85 → loss = 0.1700
  Step  90 → loss = 0.1834
  Step  95 → loss = 0.1716
  Step 100 → loss = 0.1670
  Step 105 → loss = 0.1758
  Step 110 → loss = 0.1745


In [13]:
# Rebuild loss_log from trainer history
loss_log = [(log["step"], log["loss"]) for log in trainer.state.log_history if "loss" in log]
print(f"Steps captured: {len(loss_log)}")
for step, loss in loss_log:
    print(f"  Step {step} → {loss:.4f}")

Steps captured: 22
  Step 5 → 2.2237
  Step 10 → 1.2040
  Step 15 → 0.6588
  Step 20 → 0.5412
  Step 25 → 0.4546
  Step 30 → 0.4143
  Step 35 → 0.3976
  Step 40 → 0.3538
  Step 45 → 0.3088
  Step 50 → 0.3064
  Step 55 → 0.2780
  Step 60 → 0.2508
  Step 65 → 0.2400
  Step 70 → 0.2354
  Step 75 → 0.2263
  Step 80 → 0.1853
  Step 85 → 0.1700
  Step 90 → 0.1834
  Step 95 → 0.1716
  Step 100 → 0.1670
  Step 105 → 0.1758
  Step 110 → 0.1745


In [14]:
with open("training_logs.txt", "w") as f:
    f.write("Step\tLoss\n")
    for step, loss in loss_log:
        f.write(f"{step}\t{loss:.4f}\n")

# Verify
with open("training_logs.txt", "r") as f:
    print(f.read())

Step	Loss
5	2.2237
10	1.2040
15	0.6588
20	0.5412
25	0.4546
30	0.4143
35	0.3976
40	0.3538
45	0.3088
50	0.3064
55	0.2780
60	0.2508
65	0.2400
70	0.2354
75	0.2263
80	0.1853
85	0.1700
90	0.1834
95	0.1716
100	0.1670
105	0.1758
110	0.1745



In [15]:
import shutil, os

# Save LoRA adapter weights
model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")
print("✅ Adapter saved")

# Save loss logs to txt file
with open("training_logs.txt", "w") as f:
    f.write("Step\tLoss\n")
    for step, loss in loss_log:
        f.write(f"{step}\t{loss:.4f}\n")
print("✅ training_logs.txt saved")

# Zip adapter for download
shutil.make_archive("lora_adapter", "zip", ".", "lora_adapter")
print("✅ lora_adapter.zip created")

# Confirm
print("\n📁 Files ready to download:")
for f in ["lora_adapter.zip", "training_logs.txt"]:
    size = os.path.getsize(f) / (1024*1024)
    print(f"  {f} — {size:.1f} MB")

✅ Adapter saved
✅ training_logs.txt saved
✅ lora_adapter.zip created

📁 Files ready to download:
  lora_adapter.zip — 48.8 MB
  training_logs.txt — 0.0 MB
